## CS 363M: Machine Learning Project
The goal of this project is to predict whether an aircraft wildlife strike resulted in damage to the aircraft (`INDICATED_DAMAGE = 1`) or not (`INDICATED_DAMAGE = 0`).

## Our Approach
Our work is organized as following:
1. Load and inspect the data
2. Identify missing values and data quality issues
3. Drop columns that are too sparse, redundant, or difficult to use meaningfully
4. Clean and impute remaining values
5. Encode categorical variables
6. Train and compare models
7. Evaluate performance and iterate

In [ ]:
# Standard Headers
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

## Load the Data

We begin by loading the training and test datasets and setting a few common placeholder strings such as `"UNKNOWN"`, `"Unknown"`, and `"UNK"` to missing values. Treating these consistently as null values helps us measure which values are missing more accurately and makes later cleaning steps more reliable.

In [ ]:
# Read in the data
train_data = pd.read_csv("data/train.csv", skipinitialspace=True, low_memory=False, na_values = ["UNKNOWN", "Unknown", "UNK"])
test_data = pd.read_csv("data/test.csv", skipinitialspace=True, low_memory=False, na_values=["UNKNOWN", "Unknown", "UNK"])

# Save the test ids for later use
test_ids = test_data["INDEX_NR"].copy()

# What percent of data values are null
print("Percent Null:", train_data.isnull().sum().sum() / (len(train_data) * len(train_data.columns)) * 100)

# Null amount for each column
print(train_data.isnull().sum())

## Drop Columns with Excessive Missing Values

Our first cleaning decision is to remove columns that are missing at least 50% of their values. When a feature is mostly empty, it is hard to use reliably without making assumptions during imputation.

We chose this threshold as a way to reduce noise and simplify the feature space. Rather than forcing imputation on extremely sparse columns, we keep the features that contain enough information to be useful.

Dropping a sparse column may remove some signal, but keeping too many incomplete variables can also hurt model quality and make the pipeline harder to interpret.

In [ ]:
def get_sparse_cols(df):
    sparse_cols = []

    for col in df.columns:
        if df[col].isnull().sum() >= len(df) / 2:
            sparse_cols.append(col)

    return sparse_cols

def drop_sparse_cols(df, sparse_cols):
    df = df.copy()

    # For each column, drop the column if it has 50% or more of its data missing
    print("Before:", len(df.columns))

    for col in sparse_cols:
        if col in df.columns:
            df.drop(col, inplace=True, axis=1)
            print("Dropped:", col)

    print("After:", len(df.columns))
    return df

In [ ]:
sparse_cols = get_sparse_cols(train_data)

print("Dropping sparse columns for train data")
train_data = drop_sparse_cols(train_data, sparse_cols)

print("\nDropping sparse columns for test data")
test_data = drop_sparse_cols(test_data, sparse_cols)

## Remove Redundant or Overlapping Features

Next, we remove several columns that appear to overlap heavily with other features we plan to keep. For example, some text columns have other equivalents, and some location fields overlap with latitude and longitude.

Such as:
- `AIRPORT` overlaps with `AIRPORT_ID` / coordinates
- `OPERATOR` overlaps with `OPID`
- `SPECIES` overlaps with `SPECIES_ID`

Our goal is to reduce redundancy and avoid giving the model multiple versions of the same information.

In [ ]:
def drop_redundant_cols(df):
    df = df.copy()

    print("Before:", len(df.columns))

    for col in ["AIRPORT", "OPERATOR", "SPECIES"]:
        if col in df.columns:
            df.drop(col, inplace=True, axis=1)

    print("After:", len(df.columns))
    return df

In [ ]:
print("Dropping redundant columns for train data")
train_data = drop_redundant_cols(train_data)

print("\nDropping redundant columns for test data")
test_data = drop_redundant_cols(test_data)

## Drop columns that are difficult to use or not likely to generalize well

In addition to sparse/redundant columns, we also remove a set of features that are less useful for modeling. These include:
- free-text fields
- identifiers
- administrative metadata
- columns with very limited variation
- columns that are difficult to generalize in a simple model

In [ ]:
def drop_selected_cols(df):
    df = df.copy()
    
    print("Before:", len(df.columns))

    DROP_COLS = [
        "REMARKS", "REMAINS_SENT", "COMMENTS", "SOURCE", 
        "PERSON", "LUPDATE", "TRANSFER", "NUM_STRUCK", 
        "FAAREGION", "INCIDENT_DATE", "REG", "INDEX_NR", 
        "ENG_1_POS", "ENG_2_POS", "AIRCRAFT", "AMA", 
        "AMO", "EMA", "EMO", "AC_CLASS"
    ]

    for col in DROP_COLS:
        if col in df.columns:
            df.drop(col, inplace=True, axis=1)

    print("After:", len(df.columns))
    return df

In [ ]:
print("Dropping selected columns for train data")
train_data = drop_selected_cols(train_data)

print("\nDropping selected columns for test data")
test_data = drop_selected_cols(test_data)

## Why we dropped these columns

These decisions were made to keep the feature set interpretable and to reduce noise before modeling.

In [ ]:
"""
REMARKS, COMMENTS: 
Additional notes on the incident that would be difficult to quantify within a ML model

REMAINS_SENT, SOURCE, PERSON, LUPDATE, TRANSFER, REG:
Not very predictive of if there was damage to the aircraft

NUM_STRUCK:
89% of the data was the same, and almost all of the rest was corrupted

FAAREGION:
We decided to use LONG and LAT instead

INCIDENT_DATE: 
This is less relevant than other time values that we decided to keep

INDEX_NR: 
This is a separate index and would skew our result if kept

ENG_1_POS, ENG_2_POS, AIRCRAFT, AMA, AMO, EMA, EMO:
Dropped due to a combination of many missing values and a difficulty to generalize this across different aircrafts

AC_CLASS: 
70% of the data is an aircraft, 2 records total are helicopters, rest is unknown. So this is not very helpful
"""

## Re-check missing values after removing columns

After dropping the noisiest/sparsest columns, we inspected missing values again. Since the remaining missing values are more manageable, we do imputation based on the type of variable and the context of the data.

In [ ]:
print(train_data.isnull().sum()[train_data.isnull().sum() > 0] / len(train_data) * 100)

## Cleaning and Imputing Features

Instead of applying one generic fill strategy to everything, we use more targeted methods based on the meaning of each variable.

We did the following:
- correct formatting issues in latitude and longitude
- fill missing geographic values using airport and state information
- convert time into a numeric hour feature and impute missing values using time of day groups
- fill missing distance values using phase of flight
- prepare these features so they can be used more effectively during modeling

In [ ]:
def fix_lat_long_format(df):
    df = df.copy()

    # Find the comma errors where latitude and longitude were combined
    comma_errors = df["LATITUDE"].str.contains(",", na=False)
    vals_with_com_errs = df.loc[comma_errors, "LATITUDE"].str.split(",", expand=True)

    # Fix the comma errors
    df.loc[comma_errors, "LATITUDE"] = vals_with_com_errs[0]
    df.loc[comma_errors, "LONGITUDE"] = vals_with_com_errs[1]

    return df

In [ ]:
def fill_lat_long(df, fit=True, stats=None):
    df = df.copy()

    # Make LAT and LONG numeric floats
    df["LATITUDE"] = pd.to_numeric(df["LATITUDE"], errors="coerce")
    df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")

    if fit:
        stats = {}

        for col in ["LATITUDE", "LONGITUDE"]:
            stats[col] = {
                "airport": df.groupby("AIRPORT_ID")[col].median(),
                "state": df.groupby("STATE")[col].median(),
                "global": df[col].median()
            }

    # Fill using TRAIN stats
    for col in ["LATITUDE", "LONGITUDE"]:
        df[col] = df[col].fillna(df["AIRPORT_ID"].map(stats[col]["airport"]))
        df[col] = df[col].fillna(df["STATE"].map(stats[col]["state"]))
        df[col] = df[col].fillna(stats[col]["global"])

    df.drop("AIRPORT_ID", inplace=True, axis=1, errors="ignore")
    df.drop("STATE", inplace=True, axis=1, errors="ignore")

    if fit:
        return df, stats
    return df

In [ ]:
def clean_time(df, fit=True, stats=None):
    df = df.copy()

    # Extracting out the hour value from the time column
    df["TIME"] = pd.to_datetime(df["TIME"], format="%H:%M", errors="coerce").dt.round("h").dt.hour

    if fit:
        stats = {}
        stats["time_by_period"] = df.groupby("TIME_OF_DAY")["TIME"].median()
        stats["global_time"] = df["TIME"].median()

    # Filling missing TIME values with the median value of it's TIME_OF_DAY
    df["TIME"] = df["TIME"].fillna(df["TIME_OF_DAY"].map(stats["time_by_period"]))

    # Filling rest of missing TIME values with median time
    df["TIME"] = df["TIME"].fillna(stats["global_time"])

    # Drop TIME_OF_DAY column  
    df.drop("TIME_OF_DAY", inplace=True, axis=1, errors="ignore")

    if fit:
        return df, stats
    
    return df

In [ ]:
def clean_distance(df, fit=True, stats=None):
    df = df.copy()

    if fit:
        stats = {}
        stats["phase"] = df.groupby("PHASE_OF_FLIGHT")["DISTANCE"].median()
        stats["global"] = df["DISTANCE"].median()

    # Use PHASE_OF_FLIGHT to fill in nans for DISTANCE
    df["DISTANCE"] = df["DISTANCE"].fillna(df["PHASE_OF_FLIGHT"].map(stats["phase"]))

    # Just use median DISTANCE for nans if PHASE_OF_FLIGHT is also missing for this data point
    df["DISTANCE"] = df["DISTANCE"].fillna(stats["global"])

    if fit:
        return df, stats
    
    return df

In [ ]:
def clean_features(df, fit=True, stats=None):
    df = df.copy()
    
    print("Before:", len(df.columns))

    # Fix formatting
    df = fix_lat_long_format(df)

    if fit:
        stats = {}
        df, stats["geo"] = fill_lat_long(df, fit=True)
        df, stats["time"] = clean_time(df, fit=True)
        df, stats["distance"] = clean_distance(df, fit=True)

    else:
        df = fill_lat_long(df, fit=False, stats=stats["geo"])
        df = clean_time(df, fit=False, stats=stats["time"])
        df = clean_distance(df, fit=False, stats=stats["distance"])

    print("After:", len(df.columns))
    print(df.isnull().sum() / len(df) * 100)

    if fit:
        return df, stats
    
    return df

In [ ]:
print("Cleaning features for train data")
train_data, feature_stats = clean_features(train_data, fit=True)

print("\nCleaning features for test data")
test_data = clean_features(test_data, fit=False, stats=feature_stats)

Filling remaining missing values

In [ ]:
def fill_missing_values(df, fit=True, stats=None, top_ops=None):
    df = df.copy()

    print("Before:", len(df.columns))

    # Convert runway to is or isn't airborne
    df["AIRBORNE"] = df["RUNWAY"].isnull().astype(int)
    df.drop("RUNWAY", inplace=True, axis=1, errors="ignore")

    # Fill in missing SIZE
    df["SIZE"] = df["SIZE"].fillna("UNK")
    df.drop("SPECIES_ID", inplace=True, axis=1, errors="ignore")

    if fit:
        if top_ops is None:
            top_ops = df["OPID"].value_counts().nlargest(10).index

        stats = {
            "num_engs_median": df["NUM_ENGS"].median(),
            "ac_mass_median": df["AC_MASS"].median(),
            "type_eng_mode": df["TYPE_ENG"].mode()[0]
        }

    # Reduce cardinality of OPID
    df["OPID"] = df["OPID"].where(
        df["OPID"].isin(top_ops), "UNK"
    )

    # Add missing values for PHASE_OF_FLIGHT
    df["PHASE_OF_FLIGHT"] = df["PHASE_OF_FLIGHT"].fillna("MISSING")

    # Fill in missing values using TRAIN stats
    df["NUM_ENGS"] = df["NUM_ENGS"].fillna(stats["num_engs_median"])
    df["AC_MASS"] = df["AC_MASS"].fillna(stats["ac_mass_median"])
    df["TYPE_ENG"] = df["TYPE_ENG"].fillna(stats["type_eng_mode"])

    print("After:", len(df.columns))
    print(df.isnull().sum() / len(df) * 100)

    if fit:
        return df, top_ops, stats
    return df

In [ ]:
print("Filling missing values for train data")
train_data, top_ops, fill_stats = fill_missing_values(train_data, fit=True)

print("\nFilling missing values for test data")
test_data = fill_missing_values(test_data, fit=False, stats=fill_stats, top_ops=top_ops)

## Identify Categorical Features

Before encoding, we identify which remaining columns should be treated as categorical. This matters because machine learning models generally require numeric input, but not all variables should be converted the same way.

In [ ]:
def identify_categorical_cols(df):
    categorical_cols = []
    numeric_cols = []

    for col in df.columns:
        # Try converting column to numeric
        converted = pd.to_numeric(df[col], errors='coerce')
        
        # If any portion becomes NaN, it's categorical
        num_na = converted.isnull().sum()
        
        if num_na > 1:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)

    print("Categorical Columns:")
    print(categorical_cols)

    print("\nNumeric Columns:")
    print(numeric_cols)

    print("\nTypes of the column data:")
    print(df.dtypes)

    return categorical_cols

In [ ]:
print("Identifying categorical columns from train data")
categorical_cols = identify_categorical_cols(train_data)

## Encoding Categorical Features

Initially, we encoded all categorical variables using **one-hot encoding**. This approach is simple and works well for models because it does not create any ordering between categories.

In [ ]:
def encode_cols(df, categorical_cols, fit=True, encoders=None):
    df = df.copy()

    if "INDICATED_DAMAGE" in df.columns:
        X = df.drop("INDICATED_DAMAGE", axis=1).copy()
        y = df["INDICATED_DAMAGE"].copy()
    else:
        X = df.copy()
        y = None

    print("Number of rows and cols before encoding:", X.shape)

    if fit:
        high_card_cols = []
        low_card_cols = []

        for col in categorical_cols:
            num_unique = X[col].nunique()

            # Threshold: > 10 unique values, high cardinality
            if num_unique > 10:
                high_card_cols.append(col)
            else:
                low_card_cols.append(col)

        print("High-cardinality columns (frequency encode):", high_card_cols)
        print("Low-cardinality columns (one-hot encode):", low_card_cols)

        freq_maps = {}

        # Frequency encode for high-cardinality columns
        for col in high_card_cols:
            freq = X[col].value_counts(normalize=True)
            freq_maps[col] = freq
            X[col] = X[col].map(freq)

        # One-hot encode for low-cardinality columns
        X_encoded = pd.get_dummies(X, columns=low_card_cols, drop_first=False)

        encoders = {
            "high_card_cols": high_card_cols,
            "low_card_cols": low_card_cols,
            "freq_maps": freq_maps,
            "train_cols": X_encoded.columns
        }

        print("Final shape after encoding:", X_encoded.shape)
        return X_encoded, y, encoders

    else:
        high_card_cols = encoders["high_card_cols"]
        low_card_cols = encoders["low_card_cols"]
        freq_maps = encoders["freq_maps"]
        train_cols = encoders["train_cols"]

        # Frequency encode using train frequencies
        for col in high_card_cols:
            X[col] = X[col].map(freq_maps[col]).fillna(0)

        # One-hot encode
        X_encoded = pd.get_dummies(X, columns=low_card_cols, drop_first=False)

        # Align columns to training columns
        X_encoded = X_encoded.reindex(columns=train_cols, fill_value=0)

        print("Final shape after encoding:", X_encoded.shape)
        return X_encoded

In [ ]:
print("Encoding columns for train data")
X_encoded, y, encoders = encode_cols(train_data, categorical_cols, fit=True)

print("\nEncoding columns for test data")
X_test_encoded = encode_cols(test_data, categorical_cols, fit=False, encoders=encoders)